# 서울시 도로구간 시각화 및 25m 포인트 생성

`(도로명주소)도로구간_서울/TL_SPRD_MANAGE_11_202605.shp`에서 서울시(`SIG_CD`가 `11`로 시작) 도로구간 중 도로 클래스 `3: 로`, `4: 길`만 추출해 단색으로 시각화하고, 도로 라인 위에 25m 간격 포인트를 생성합니다.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from silverwalk import accidents, business, config, features, population, roads, social_welfare, speed

# 노트북 커널이 이전 버전의 모듈을 캐시할 수 있으므로, 실행할 때마다 최신 파일을 다시 불러옵니다.
for module in [config, roads, features, accidents, speed, business, population, social_welfare]:
    importlib.reload(module)

from silverwalk.accidents import add_accident_risk
from silverwalk.business import add_business_category_counts
from silverwalk.config import (
    ACCIDENT_BUFFER_M,
    ACCIDENT_CSV,
    BUSINESS_CATEGORY_COLUMNS,
    BUSINESS_CSV,
    BUSINESS_RADIUS_M,
    ELDERLY_POPULATION_COLUMN,
    NODELINK_SHP,
    OUTPUT_JOIN_CSV,
    POINT_INTERVAL_M,
    POPULATION_GRID_DIR,
    POPULATION_RADIUS_M,
    SOCIAL_WELFARE_COLUMN,
    SOCIAL_WELFARE_GEOCODED_CSV,
    SOCIAL_WELFARE_RADIUS_M,
    SOCIAL_WELFARE_XLSX,
    TARGET_REGION_NAME,
    VWORLD_API_KEY_ENV,
)
from silverwalk.features import make_base_point_df
from silverwalk.population import add_elderly_population_300m
from silverwalk.social_welfare import add_social_welfare_facility_counts
from silverwalk.roads import (
    create_point_buffers,
    create_road_points,
    filter_target_roads,
    load_road_segments,
    make_road_summary,
)
from silverwalk.speed import add_speed_limit

# 한글 폰트가 설치된 환경에서는 한글 제목/범례가 정상 표시됩니다.
plt.rcParams["font.family"] = ["NanumGothic", "Malgun Gothic", "AppleGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
roads, ROAD_SHP = load_road_segments()

print(f"도로구간 파일: {ROAD_SHP}")
print(f"전체 도로구간 수: {len(roads):,}")
print(f"원본 좌표계: {roads.crs}")
display(roads.head())


In [ ]:
target_all_roads, target_roads = filter_target_roads(roads)
summary = make_road_summary(target_all_roads, target_roads)

display(summary)
display(target_roads["ROA_CLS_SE"].value_counts().sort_index().rename("도로 클래스별 구간 수"))
display(target_roads[["SIG_CD", "RN", "RN_CD", "ROAD_LT", "ROAD_BT", "ROA_CLS_SE", "geometry"]].head())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

target_roads.plot(
    ax=ax,
    linewidth=0.45,
    color="#2563eb",
    alpha=0.85,
)

ax.set_title(f"{TARGET_REGION_NAME} Road Network - Classes 3 and 4", fontsize=16, pad=12)
ax.set_axis_off()
ax.set_aspect("equal")

plt.tight_layout()
plt.show()


## 서울시 도로 25m 간격 포인트 생성

현재 도로 데이터 좌표계는 EPSG:5179이므로 거리 단위는 미터입니다. 아래 코드는 도로 클래스 `3: 로`, `4: 길` 라인 위에 25m 간격 포인트를 생성합니다.


In [ ]:
points_25m_gdf = create_road_points(target_roads, distance=POINT_INTERVAL_M)

print(f"생성된 25m 간격 포인트 수: {len(points_25m_gdf):,}")
display(points_25m_gdf.head())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

target_roads.plot(
    ax=ax,
    linewidth=0.35,
    color="#94a3b8",
    alpha=0.7,
)

points_25m_gdf.plot(
    ax=ax,
    markersize=0.25,
    color="#dc2626",
    alpha=0.75,
)

ax.set_title(f"{TARGET_REGION_NAME} Road Points at 25m Intervals - Classes 3 and 4", fontsize=16, pad=12)
ax.set_axis_off()
ax.set_aspect("equal")

plt.tight_layout()
plt.show()


## 25m 포인트별 50m 버퍼 생성

`points_25m_gdf`의 각 포인트를 중심으로 반경 50m 버퍼를 생성합니다. EPSG:5179 좌표계는 미터 단위이므로 `buffer(50)`을 사용합니다.

In [ ]:
point_buffers_50m_gdf = create_point_buffers(points_25m_gdf, buffer_distance_m=ACCIDENT_BUFFER_M)

print(f"생성된 50m 버퍼 수: {len(point_buffers_50m_gdf):,}")
display(point_buffers_50m_gdf.head())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

point_buffers_50m_gdf.plot(
    ax=ax,
    color="#f97316",
    alpha=0.12,
    edgecolor="none",
)

target_roads.plot(
    ax=ax,
    linewidth=0.25,
    color="#334155",
    alpha=0.75,
)

points_25m_gdf.plot(
    ax=ax,
    markersize=0.15,
    color="#dc2626",
    alpha=0.6,
)

ax.set_title(f"50m Buffers Around {TARGET_REGION_NAME} 25m Road Points - Classes 3 and 4", fontsize=16, pad=12)
ax.set_axis_off()
ax.set_aspect("equal")

plt.tight_layout()
plt.show()


## Feature 추가 함수

Feature 추가 함수는 `src/silverwalk/` 모듈로 분리했습니다. 이 노트북에서는 결합 순서만 실행합니다.


In [ ]:
print(f"상가 업종 컬럼 수: {len(BUSINESS_CATEGORY_COLUMNS):,}")
print(f"사고 데이터: {ACCIDENT_CSV}")
print(f"ITS 표준노드링크: {NODELINK_SHP}")
print(f"상가 정보: {BUSINESS_CSV}")
print(f"고령인구 격자 데이터: {POPULATION_GRID_DIR}")
print(f"노인의료복지시설 엑셀: {SOCIAL_WELFARE_XLSX}")
print(f"VWorld 인증키 환경변수명: {VWORLD_API_KEY_ENV}")
print(f"노인의료복지시설 좌표 CSV: {SOCIAL_WELFARE_GEOCODED_CSV} (직접 좌표 CSV를 사용할 때만 필요)")


## 최종 CSV 생성 및 저장

기본 포인트 데이터에 feature 추가 함수들을 순서대로 실행한 뒤 CSV로 저장합니다.

In [ ]:
# 25m 포인트
final_join_df = make_base_point_df(points_25m_gdf)

# 위험도 컬럼 추가
final_join_df = add_accident_risk(
    final_df=final_join_df,
    points_gdf=points_25m_gdf,
    point_buffers_gdf=point_buffers_50m_gdf,
    accident_csv=ACCIDENT_CSV,
)

# 제한속도 컬럼 추가
final_join_df = add_speed_limit(
    final_df=final_join_df,
    points_gdf=points_25m_gdf,
    nodelink_shp=NODELINK_SHP,
)

# 반경 300m 업종별 상가 수 컬럼 추가
final_join_df = add_business_category_counts(
    final_df=final_join_df,
    points_gdf=points_25m_gdf,
    business_csv=BUSINESS_CSV,
    radius_m=BUSINESS_RADIUS_M,
)

# 반경 300m 65세 이상 거주인구 컬럼 추가
final_join_df = add_elderly_population_300m(
    final_df=final_join_df,
    points_gdf=points_25m_gdf,
    radius_m=POPULATION_RADIUS_M,
)

# 반경 300m 노인의료복지시설 수 컬럼 추가
# VWorld 인증키는 환경변수 VWORLD_API_KEY에 설정하거나 vworld_api_key 인자로 직접 전달합니다.
final_join_df = add_social_welfare_facility_counts(
    final_df=final_join_df,
    points_gdf=points_25m_gdf,
    facility_xlsx=SOCIAL_WELFARE_XLSX,
    geocoded_csv=SOCIAL_WELFARE_GEOCODED_CSV,
    radius_m=SOCIAL_WELFARE_RADIUS_M,
    vworld_api_key=None,
    use_geocoded_csv=False,
)

# 마무리
final_join_df = final_join_df[["POINT_ID", "위도", "경도", "제한속도", "위험도", ELDERLY_POPULATION_COLUMN, SOCIAL_WELFARE_COLUMN] + BUSINESS_CATEGORY_COLUMNS]

OUTPUT_JOIN_CSV.parent.mkdir(parents=True, exist_ok=True)
final_join_df.to_csv(OUTPUT_JOIN_CSV, index=False, encoding="utf-8-sig")

print(f"저장된 최종 데이터 행 수: {len(final_join_df):,}")
print(f"제한속도 결측 수: {final_join_df['제한속도'].isna().sum():,}")
print(f"상가 업종 컬럼 수: {len(BUSINESS_CATEGORY_COLUMNS):,}")
print(f"고령인구 컬럼: {ELDERLY_POPULATION_COLUMN}")
print(f"사회복지시설 컬럼: {SOCIAL_WELFARE_COLUMN}")
print(f"CSV 저장: {OUTPUT_JOIN_CSV}")
display(final_join_df.head())
